In [14]:
from langgraph.graph import StateGraph , START, END
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from pydantic import BaseModel, Field
from typing import TypedDict, List, Dict, Literal
from dotenv import load_dotenv
load_dotenv()
model = ChatOpenAI(model="gpt-4o-mini")

In [ ]:
class sentiment(BaseModel):
    sentiment:Literal['pos','neg']=Field(description="Sentimentfinding")

stroutputmodel=model.with_structured_output(sentiment)

class diagnosis(BaseModel):
    issueType:Literal['UX','software','hardware']=Field(description="Issuetype")
    userTone:Literal["Angry","normal","Urgent"]=Field(description="Usertone")

strmodel_2=model.with_structured_output(diagnosis)

class SentimentAnalysis(TypedDict):
    review:str
    sentiment:str
    diagnosis:dict
    response:str

def FindSentiment(state:SentimentAnalysis):
    prompt=PromptTemplate(
        template="find the sentiment for the given review: {review}",
        input_variables=['review']
    )
    chain=prompt|stroutputmodel
    result=chain.invoke({'review':state['review']})
    return {'sentiment':result.sentiment}

def conditionCheck(state:SentimentAnalysis)->Literal['FindDignosis','positivefeedback']:
    if state['sentiment']=='pos':
        return 'positivefeedback'
    else:
        return 'FindDignosis'

def FindDignosis(state:SentimentAnalysis):
    prompt=PromptTemplate(
        template="Find the issuetype and  usertone for the given review:{review}",
        input_variables=['review']
    )
    chain=prompt|strmodel_2
    output=chain.invoke({'review':state['review']})
    return {'diagnosis':output.model_dump()}

def positivefeedback(state:SentimentAnalysis):
    prompt=f"Generate the positive AI response for the review:{state['review']}"
    result=model.invoke(prompt)
    return{'response':result.content}

def negativefeedback(state:SentimentAnalysis):
    prompt=f"Generate the negative AI response for the review{state['review']}"
    result=model.invoke(prompt)
    return{'response':result.content}

graph=StateGraph(SentimentAnalysis)
graph.add_node('FindSentiment',FindSentiment)

graph.add_node('FindDignosis',FindDignosis)
graph.add_node('positivefeedback',positivefeedback)
graph.add_node('negativefeedback',negativefeedback)

graph.add_edge(START,'FindSentiment')
graph.add_conditional_edges('FindSentiment',conditionCheck)
graph.add_edge('FindDignosis','negativefeedback')
graph.add_edge('negativefeedback',END)
graph.add_edge('positivefeedback',END)

graph.compile()
workflow=graph.compile()
initial_state={'review':'I dont like this mobile'}
final_state=workflow.invoke(initial_state)
print(final_state)


{'review': 'I dont like this mobile', 'sentiment': 'neg', 'response': "We're sorry to hear that you're not satisfied with the mobile. We appreciate your feedback and would like to understand more about your experience. If you have specific issues or concerns, please share them with us so we can assist you better. Thank you for your input!"}
